In [1]:
pip install tensorflow tensorflow-hub librosa pandas scikit-learn tqdm soundfile


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -U "tensorflow<2.16" "keras<3" tensorflow-hub==0.16.1


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow<2.16 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow<2.16


In [3]:
import os
import tensorflow as tf
import tensorflow_hub as hub
import pandas as pd
import numpy as np
import librosa
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models

In [4]:
base_dir = r"C:/Users/SiluniR/Documents/final research/dataset/ESC-50/categorized_ESC50/split/splits"
train_csv = os.path.join(base_dir, "train.csv")
val_csv   = os.path.join(base_dir, "val.csv")
test_csv  = os.path.join(base_dir, "test.csv")

sr_target = 16000
yamnet_model_handle = "https://tfhub.dev/google/yamnet/1"

In [5]:
# === LOAD CSVs ===
train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
test_df  = pd.read_csv(test_csv)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 7785, Val: 973, Test: 974


In [6]:
# === ENCODE LABELS ===
label_encoder = LabelEncoder()
label_encoder.fit(train_df["label"])
n_classes = len(label_encoder.classes_)
print("✅ Classes:", n_classes, label_encoder.classes_)

✅ Classes: 55 ['air_conditioner' 'airplane' 'breathing' 'brushing_teeth' 'can_opening'
 'car_horn' 'cat' 'chainsaw' 'children_playing' 'chirping_birds'
 'church_bells' 'clapping' 'clock_alarm' 'clock_tick' 'coughing' 'cow'
 'crackling_fire' 'crickets' 'crow' 'crying_baby' 'dog' 'door_wood_creaks'
 'door_wood_knock' 'drilling' 'drinking_sipping' 'engine' 'fireworks'
 'footsteps' 'frog' 'glass_breaking' 'gun_shot' 'hand_saw' 'helicopter'
 'hen' 'insects' 'keyboard_typing' 'laughing' 'mouse_click' 'pig'
 'pouring_water' 'rain' 'rooster' 'sea_waves' 'sheep' 'siren' 'sneezing'
 'snoring' 'street_music' 'thunderstorm' 'toilet_flush' 'train'
 'vacuum_cleaner' 'washing_machine' 'water_drops' 'wind']


In [7]:
# === LOAD PRETRAINED YAMNET ===
yamnet_model = hub.load(yamnet_model_handle)

def extract_embedding(file_path):
    """Convert audio → YAMNet embedding"""
    try:
        waveform, _ = librosa.load(file_path, sr=sr_target, mono=True)
        scores, embeddings, spectrogram = yamnet_model(waveform)
        mean_emb = tf.reduce_mean(embeddings, axis=0)
        return mean_emb.numpy()
    except Exception as e:
        print("⚠️ Error:", e)
        return np.zeros((1024,))

In [8]:
# === BUILD EMBEDDING MATRICES ===
def build_dataset(df):
    X, y = [], []
    for _, row in df.iterrows():
        X.append(extract_embedding(row["filepath"]))
        y.append(label_encoder.transform([row["label"]])[0])
    return np.array(X), tf.keras.utils.to_categorical(y, num_classes=n_classes)

print("🎧 Extracting embeddings (this may take a while)...")
X_train, y_train = build_dataset(train_df)
X_val, y_val     = build_dataset(val_df)
X_test, y_test   = build_dataset(test_df)

print("Shapes:", X_train.shape, y_train.shape)

🎧 Extracting embeddings (this may take a while)...


C:\Users\SiluniR\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


Shapes: (7785, 1024) (7785, 55)


In [9]:
# === BUILD CLASSIFIER ===
model = models.Sequential([
    layers.Input(shape=(1024,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(n_classes, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [10]:
# === TRAIN ===
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=30,
                    batch_size=32)


Epoch 1/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.4909 - loss: 2.4286 - val_accuracy: 0.6598 - val_loss: 1.5080
Epoch 2/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.6717 - loss: 1.3805 - val_accuracy: 0.7246 - val_loss: 1.1268
Epoch 3/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7162 - loss: 1.0997 - val_accuracy: 0.7472 - val_loss: 0.9387
Epoch 4/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.7543 - loss: 0.9493 - val_accuracy: 0.7739 - val_loss: 0.8312
Epoch 5/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.7679 - loss: 0.8546 - val_accuracy: 0.7924 - val_loss: 0.7464
Epoch 6/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.7859 - loss: 0.7759 - val_accuracy: 0.8109 - val_loss: 0.6876
Epoch 7/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.7979 - loss: 0.7266 - val_accuracy: 0.8212 - val_loss: 0.6487
Epoch 8/30
244/244 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8101 - loss: 0.6836 - val_acc

In [11]:
# === EVALUATE ===
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print(f"✅ Test Accuracy: {test_acc:.4f}")

31/31 - 0s - 16ms/step - accuracy: 0.8891 - loss: 0.3911
✅ Test Accuracy: 0.8891


In [12]:
export_path = r"C:\Users\SiluniR\Documents\final research\dataset\ESC-50\hazard_yamnet.keras"
model.save(export_path)
print("✅ Model saved as:", export_path)


✅ Model saved as: C:\Users\SiluniR\Documents\final research\dataset\ESC-50\hazard_yamnet.keras


In [13]:
from tensorflow.keras import optimizers, callbacks
import tensorflow as tf

# === LOAD TRAINED MODEL ===
model_path = r"C:\Users\SiluniR\Documents\final research\dataset\ESC-50\hazard_yamnet.keras"
model = tf.keras.models.load_model(model_path, compile=False)
print("✅ Model loaded successfully from:", model_path)

# === UNFREEZE DEEPER LAYERS ===
# Optional: unfreeze last N layers if you had frozen them before
# Here we unfreeze last 60 layers (typical for YAMNet fine-tuning)
for layer in model.layers[-60:]:
    layer.trainable = True

# === RECOMPILE WITH SMALLER LR ===
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# === CALLBACKS FOR STABLE TRAINING ===
early_stop = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
reduce_lr  = callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6)
checkpoint = callbacks.ModelCheckpoint(
    "yamnet_best_finetuned.keras", save_best_only=True, monitor="val_accuracy", mode="max"
)

# === CONTINUE TRAINING (FINE-TUNING) ===
history_finetune = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16,
    callbacks=[early_stop, reduce_lr, checkpoint]
)


# === SAVE FINAL FINE-TUNED MODEL ===
save_path = r"C:\Users\SiluniR\Documents\final research\dataset\ESC-50\hazard_yamnet_finetuned_stage2.keras"
model.save(save_path)
print("💾 Fine-tuned model saved successfully at:", save_path)


✅ Model loaded successfully from: C:\Users\SiluniR\Documents\final research\dataset\ESC-50\hazard_yamnet.keras
Epoch 1/10
487/487 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.9107 - loss: 0.3008 - val_accuracy: 0.8921 - val_loss: 0.3863 - learning_rate: 1.0000e-05
Epoch 2/10
487/487 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9101 - loss: 0.2921 - val_accuracy: 0.8962 - val_loss: 0.3771 - learning_rate: 1.0000e-05
Epoch 3/10
487/487 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.9110 - loss: 0.2863 - val_accuracy: 0.8962 - val_loss: 0.3794 - learning_rate: 1.0000e-05
Epoch 4/10
487/487 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.9133 - loss: 0.2855 - val_accuracy: 0.8993 - val_loss: 0.3729 - learning_rate: 1.0000e-05
Epoch 5/10
487/487 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9146 - loss: 0.2855 - val_accuracy: 0.8972 - val_loss: 0.3764 - learning_rate: 1.0000e-05
Epoch 6/10
487/487 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.9179 - loss: 0.2834 - val_accuracy:

In [28]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print(f"✅ Test Accuracy: {test_acc:.4f}")

31/31 - 0s - 4ms/step - accuracy: 0.8943 - loss: 0.3678
✅ Test Accuracy: 0.8943
